In [ ]:
import pandas as pd

df = pd.read_pickle("reviews_gpu.pkl")

df = df[['review', 'label', 'aspects']]

df['label'] = df['label'].map({-1: 0, 1: 1})

df.head()

,review,label,aspects
0,put this movie out of its misery and burn the ...,0,"[(movie, whole), (bit, least), (majority, vast..."
1,nice mobile phone thank you flipkart for fast ...,1,"[(phone, nice), (phone, mobile), (delivery, fa..."
2,while the finished producta working phone with...,0,[]
3,best price phone,1,"[(phone, good)]"
4,this book was a requirement for my th grade gi...,0,[]


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df[['review', 'label']],
    test_size=0.1,
    random_state=42
)

In [ ]:


import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["review"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 9920/9920 [00:01<00:00, 6613.99 examples/s] 


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaForSequenceClassification: ['lm_head.dense.weight', 'roberta.pooler.dense.bias', 'lm_head.dense.bias', 'roberta.pooler.dense.weight', 'lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.o

In [9]:
training_args = TrainingArguments(
    output_dir="./xlmr_absa",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\transformers\optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  3%|▎         | 500/16740 [03:29<1:41:07,  2.68it/s]

{'loss': 0.4124, 'learning_rate': 1.94026284348865e-05, 'epoch': 0.09}


  6%|▌         | 1000/16740 [06:47<1:41:00,  2.60it/s]

{'loss': 0.2943, 'learning_rate': 1.8805256869773e-05, 'epoch': 0.18}


  9%|▉         | 1500/16740 [10:27<1:38:55,  2.57it/s] 

...

                                                       
100%|██████████| 16740/16740 [1:35:01<00:00,  2.80it/s]

{'eval_loss': 0.2917402982711792, 'eval_runtime': 51.9858, 'eval_samples_per_second': 190.821, 'eval_steps_per_second': 11.926, 'epoch': 3.0}


c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\transformers\trainer.py:2254: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(best_model_p

{'train_runtime': 5707.7749, 'train_samples_per_second': 46.921, 'train_steps_per_second': 2.933, 'train_loss': 0.20753334824756908, 'epoch': 3.0}


TrainOutput(global_step=16740, training_loss=0.20753334824756908, metrics={'train_runtime': 5707.7749, 'train_samples_per_second': 46.921, 'train_steps_per_second': 2.933, 'train_loss': 0.20753334824756908, 'epoch': 3.0})

In [9]:
trainer.save_model("./xlmr_absa_final")
tokenizer.save_pretrained("./xlmr_absa_final")

('./xlmr_absa_final\\tokenizer_config.json',
 './xlmr_absa_final\\special_tokens_map.json',
 './xlmr_absa_final\\tokenizer.json')

In [10]:
tokenizer = AutoTokenizer.from_pretrained("./xlmr_absa_final")
model = AutoModelForSequenceClassification.from_pretrained("./xlmr_absa_final")

model.eval()

c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\transformers\modeling_utils.py:463: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_f

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768,

In [ ]:


import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("./xlmr_absa_final")
model = AutoModelForSequenceClassification.from_pretrained("./xlmr_absa_final")
model.to(device)
model.eval()



df = pd.read_pickle("reviews_gpu.pkl")
df = df[['review', 'label', 'aspects']]
df['label'] = df['label'].map({-1: 0, 1: 1})



df_expanded = df.copy()

df_expanded['orig_index'] = df_expanded.index

df_expanded = df_expanded.explode('aspects')

df_expanded = df_expanded[df_expanded['aspects'].notna()]

df_expanded[['aspect', 'opinion']] = pd.DataFrame(
    df_expanded['aspects'].tolist(),
    index=df_expanded.index
)

df_expanded = df_expanded.drop(columns=['aspects'])

df_expanded = df_expanded[df_expanded['aspect'].notna()]

df_expanded['review'] = df_expanded['review'].astype(str)
df_expanded['aspect'] = df_expanded['aspect'].astype(str)

print("Expanded dataset size:", len(df_expanded))


class ABSADataset(Dataset):
    def __init__(self, reviews, aspects, tokenizer, max_len=128):
        self.encodings = tokenizer(
            reviews,
            aspects,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])


dataset = ABSADataset(
    df_expanded['review'].tolist(),
    df_expanded['aspect'].tolist(),
    tokenizer
)

loader = DataLoader(dataset, batch_size=32, shuffle=False)


all_preds = []

with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        probs = F.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())



df_expanded['sentiment'] = [
    "positive" if p == 1 else "negative"
    for p in all_preds
]


final_df = df_expanded.groupby('orig_index').apply(
    lambda x: list(zip(x['aspect'], x['sentiment']))
)

df['aspect_sentiments'] = None
df.loc[final_df.index, 'aspect_sentiments'] = final_df



c:\Users\Harsh Panchal\anaconda3\envs\ABSA\lib\site-packages\transformers\modeling_utils.py:463: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_f

Expanded dataset size: 193404
Inference Completed Correctly ✅


In [27]:
df['aspect_sentiments'].isna().sum()

31279

In [28]:
31279/99191

0.31534110957647365

In [29]:
df.to_pickle("final_absa_dataset.pkl")

In [30]:
df

,review,label,aspects,aspect_sentiments
0,put this movie out of its misery and burn the ...,0,"[(movie, whole), (bit, least), (majority, vast...","[(movie, negative), (bit, negative), (majority..."
1,nice mobile phone thank you flipkart for fast ...,1,"[(phone, nice), (phone, mobile), (delivery, fa...","[(phone, positive), (phone, positive), (delive..."
2,while the finished producta working phone with...,0,[],None
3,best price phone,1,"[(phone, good)]","[(phone, positive)]"
4,this book was a requirement for my th grade gi...,0,[],None
...,...,...,...,...
99259,it would be hard to describe the plot of this ...,1,"[(sense, great)]","[(sense, positive)]"
99260,good product but battery drain faster when int...,1,"[(product, good)]","[(product, positive)]"
99261,i was disappointed in this documentaryi though...,0,"[(player, great), (blue, deep), (it, dry)]","[(player, negative), (blue, negative), (it, ne..."
99262,i am pretty happy with installment out of the ...,1,[],None


In [ ]:

if isinstance(df["aspect_sentiments"].iloc[0], str):
    df["aspect_sentiments"] = df["aspect_sentiments"].apply(ast.literal_eval)


df["aspect_sentiments"] = df["aspect_sentiments"].apply(
    lambda x: x if isinstance(x, list) and len(x) > 0 else None
)

df_exploded = df.explode("aspect_sentiments")

df_exploded["aspect"] = df_exploded["aspect_sentiments"].apply(
    lambda x: x[0] if isinstance(x, tuple) else "General"
)

df_exploded["aspect_sentiment"] = df_exploded["aspect_sentiments"].apply(
    lambda x: x[1] if isinstance(x, tuple) else None
)

sentiment_map = {
    "positive": 1,
    "negative": 0,
    "Positive": 1,
    "Negative": 0
}

df_exploded["aspect_sentiment"] = df_exploded["aspect_sentiment"].replace(sentiment_map)

df_exploded["aspect_sentiment"] = df_exploded["aspect_sentiment"].fillna(
    df_exploded["label"]
)

df_exploded["aspect_sentiment"] = df_exploded["aspect_sentiment"].astype(int)


final_df = df_exploded[["review", "aspect", "aspect_sentiment"]].reset_index(drop=True)

print("Final shape:", final_df.shape)
print(final_df.head())


final_df.to_pickle("final_absa_dataset.pkl")
